# CNN para Detecção de Câncer de Mama em Mamografias

**Dataset:** CBIS-DDSM (Curated Breast Imaging Subset of DDSM)  
**Plataforma:** Google Colab (GPU T4)  
**Framework:** TensorFlow / Keras

## Objetivo

Treinar uma Rede Neural Convolucional (CNN) com *transfer learning* para classificar mamografias como **benignas** ou **malignas**, e comparar explicitamente dois regimes de compilação:

| Setup | Loss | Métrica |
|-------|------|---------|
| **A** | `binary_crossentropy` | `Recall` |
| **B** | `sparse_categorical_crossentropy` | `accuracy` |

A escolha de loss e métrica é uma decisão clínica: em triagem médica, **um falso negativo (maligno classificado como benigno) tem custo muito maior** que um falso positivo. Por isso, o Setup A é o mais adequado para este problema.

> ⚠️ **Aviso:** Este modelo é um protótipo acadêmico. Não substitui diagnóstico médico.

---
## Etapa 1 — Configuração do Ambiente e Download dos Dados

O Google Colab não tem armazenamento persistente. O dataset CBIS-DDSM (~12 GB) é baixado diretamente do Kaggle via API autenticada a cada sessão.

**Pré-requisito:** você precisa de um arquivo `kaggle.json` com suas credenciais. Obtenha em kaggle.com → Account → API → Create New Token.

In [ ]:
# Instalar biblioteca do Kaggle
!pip install -q kaggle

# Fazer upload do arquivo access_token (contém apenas a chave da API)
from google.colab import files
print("Faça o upload do seu arquivo 'access_token':")
uploaded = files.upload()

# Ler a chave do arquivo enviado
token_filename = list(uploaded.keys())[0]
with open(token_filename, 'r') as f:
    content = f.read().strip()

# Verificar se já é um JSON completo ou apenas a chave
import json, os

try:
    token_data = json.loads(content)
    # Já é JSON — verificar se tem username
    if 'username' not in token_data:
        kaggle_username = input("Digite seu usuário do Kaggle (kaggle.com/seu-usuario): ").strip()
        token_data['username'] = kaggle_username
except json.JSONDecodeError:
    # Arquivo contém apenas a chave (string pura)
    kaggle_username = input("Digite seu usuário do Kaggle (kaggle.com/seu-usuario): ").strip()
    token_data = {"username": kaggle_username, "key": content}

# Salvar kaggle.json no local esperado pela biblioteca
os.makedirs(os.path.expanduser('~/.kaggle'), exist_ok=True)
kaggle_json_path = os.path.expanduser('~/.kaggle/kaggle.json')
with open(kaggle_json_path, 'w') as f:
    json.dump(token_data, f)
os.chmod(kaggle_json_path, 0o600)

print(f"Credencial configurada: usuário='{token_data['username']}', chave='{token_data['key'][:6]}...' ✓")

In [ ]:
# Baixar e extrair o dataset CBIS-DDSM
# O dataset contém imagens PNG de mamografias + CSVs de metadados
!kaggle datasets download -d awsaf49/cbis-ddsm-breast-cancer-image-dataset \
    --unzip -p /content/cbis

# Listar arquivos extraídos
!ls /content/cbis/

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
import os
import warnings
warnings.filterwarnings('ignore')

from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, roc_curve
)
from sklearn.utils.class_weight import compute_class_weight

print(f"TensorFlow versão: {tf.__version__}")
print(f"GPU disponível: {tf.config.list_physical_devices('GPU')}")

# Seed para reprodutibilidade
SEED = 42
tf.random.set_seed(SEED)
np.random.seed(SEED)

---
## Etapa 2 — Exploração dos Metadados (EDA)

O CBIS-DDSM organiza os dados em dois tipos de lesão:
- **Calcificação** (`calc`): depósitos de cálcio nas células mamárias
- **Massa** (`mass`): crescimento anormal de tecido

Cada tipo tem CSVs separados para treino e teste. Vamos unificá-los em um único DataFrame.

A coluna `pathology` pode conter três valores:
- `BENIGN` → classe 0
- `BENIGN_WITHOUT_CALLBACK` → classe 0 (benigno confirmado sem necessidade de acompanhamento)
- `MALIGNANT` → classe 1

In [ ]:
BASE_PATH = '/content/cbis'

# Carregar os 4 CSVs de metadados
calc_train = pd.read_csv(f'{BASE_PATH}/csv/calc_case_description_train_set.csv')
calc_test  = pd.read_csv(f'{BASE_PATH}/csv/calc_case_description_test_set.csv')
mass_train = pd.read_csv(f'{BASE_PATH}/csv/mass_case_description_train_set.csv')
mass_test  = pd.read_csv(f'{BASE_PATH}/csv/mass_case_description_test_set.csv')

# Adicionar coluna de tipo de lesão para rastreabilidade
calc_train['lesion_type'] = 'calc'
calc_test['lesion_type']  = 'calc'
mass_train['lesion_type'] = 'mass'
mass_test['lesion_type']  = 'mass'

# Unificar treino e teste em DataFrames únicos
df_train = pd.concat([calc_train, mass_train], ignore_index=True)
df_test  = pd.concat([calc_test,  mass_test],  ignore_index=True)

print(f"Treino: {len(df_train)} amostras")
print(f"Teste:  {len(df_test)} amostras")
print(f"\nColunas disponíveis:\n{df_train.columns.tolist()}")

In [ ]:
# Mapear pathology para rótulo binário
def map_label(pathology):
    return 0 if 'BENIGN' in str(pathology) else 1

df_train['label'] = df_train['pathology'].apply(map_label)
df_test['label']  = df_test['pathology'].apply(map_label)

# ─── Descobrir estrutura real das imagens no disco ───────────────────────────
print("Listando estrutura extraída em /content/cbis ...")
!ls /content/cbis/

print("\nPrimeiras imagens encontradas (PNG/JPG):")
!find /content/cbis -maxdepth 4 \( -name "*.png" -o -name "*.jpg" \) | head -20

In [ ]:
# ─── Construir mapeamento: patient_id → caminho real da imagem ───────────────
# O Kaggle organiza as imagens em pastas por paciente/lateralidade,
# mas os CSVs referenciam caminhos DICOM originais (*.dcm) que não existem.
# Aqui indexamos todos os PNGs/JPGs encontrados e linkamos via patient info.

import glob

# Coletar todos os arquivos de imagem disponíveis
all_images = (
    glob.glob('/content/cbis/**/*.png', recursive=True) +
    glob.glob('/content/cbis/**/*.jpg', recursive=True)
)
print(f"Total de imagens encontradas: {len(all_images)}")

# Extrair identificador do paciente a partir do caminho do arquivo
# Exemplo de path real: /content/cbis/jpeg/1.3.6.../000000.jpg
# Exemplo de path no CSV: Calc-Training_P_00005_RIGHT_CC/1.3.6.../1.3.6.../000000.dcm
# A chave de ligação é o diretório pai imediato (ID do estudo DICOM)

# Indexar por: (nome_da_pasta_do_estudo)
img_index = {}
for path in all_images:
    # O penúltimo componente do path é o ID do estudo
    study_id = os.path.basename(os.path.dirname(path))
    img_index[study_id] = path

print(f"IDs de estudo indexados: {len(img_index)}")
print("Exemplo de chave:", list(img_index.keys())[0])

def resolve_path(csv_path):
    """Extrai o ID do estudo do caminho CSV e busca o arquivo real."""
    parts = str(csv_path).strip().replace('\\', '/').split('/')
    # O CSV tem: pasta_paciente / id_serie / id_estudo / 000000.dcm
    # Tentamos o id_estudo (penúltimo antes do arquivo)
    for part in reversed(parts[:-1]):
        if part in img_index:
            return img_index[part]
    return None  # não encontrado

df_train['full_path'] = df_train['image file path'].apply(resolve_path)
df_test['full_path']  = df_test['image file path'].apply(resolve_path)

# Remover linhas sem imagem correspondente
n_before = len(df_train)
df_train = df_train.dropna(subset=['full_path']).reset_index(drop=True)
df_test  = df_test.dropna(subset=['full_path']).reset_index(drop=True)

print(f"\nTreino: {n_before} → {len(df_train)} amostras com imagem encontrada")
print(f"Teste:  {len(df_test)} amostras com imagem encontrada")
print("\nExemplo de caminho resolvido:")
print(df_train['full_path'].iloc[0])

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Gráfico 1: Distribuição de classes
counts = df_train['label'].value_counts()
axes[0].bar(['Benigno (0)', 'Maligno (1)'], counts.values,
            color=['steelblue', 'tomato'], edgecolor='black')
axes[0].set_title('Distribuição de Classes — Treino', fontsize=13)
axes[0].set_ylabel('Número de amostras')
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 5, str(v), ha='center', fontweight='bold')

# Gráfico 2: Proporção por tipo de lesão
pivot = df_train.groupby(['lesion_type', 'label']).size().unstack(fill_value=0)
pivot.columns = ['Benigno', 'Maligno']
pivot.plot(kind='bar', ax=axes[1], color=['steelblue', 'tomato'],
           edgecolor='black', rot=0)
axes[1].set_title('Distribuição por Tipo de Lesão — Treino', fontsize=13)
axes[1].set_xlabel('Tipo de lesão')
axes[1].set_ylabel('Número de amostras')
axes[1].legend()

plt.tight_layout()
plt.show()

# Proporção geral
total = len(df_train)
maligno = df_train['label'].sum()
print(f"\nDesbalanceamento: {(total-maligno)/total*100:.1f}% benigno | {maligno/total*100:.1f}% maligno")
print("→ Usaremos class_weight para compensar o desbalanceamento durante o treino.")

In [ ]:
# Visualizar amostras de cada classe
import cv2

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
fig.suptitle('Amostras do Dataset CBIS-DDSM', fontsize=14, fontweight='bold')

for col, (label, name, color) in enumerate([(0, 'Benigno', 'steelblue'),
                                              (1, 'Maligno', 'tomato')]):
    samples = df_train[df_train['label'] == label].sample(4, random_state=SEED)
    for row, (_, row_data) in enumerate(samples.iterrows()):
        img = cv2.imread(row_data['full_path'], cv2.IMREAD_GRAYSCALE)
        if img is not None:
            axes[col][row].imshow(img, cmap='gray')
            axes[col][row].set_title(f'{name}\n{row_data["lesion_type"]}',
                                     color=color, fontsize=10)
        axes[col][row].axis('off')

plt.tight_layout()
plt.show()

---
## Etapa 3 — Pré-processamento e Pipeline de Dados

### Por que 224×224?
O EfficientNetB0 foi originalmente treinado com imagens 224×224. Usar a mesma resolução maximiza o aproveitamento dos pesos pré-treinados, que aprenderam a detectar features nessa escala.

### Por que normalizar para [0, 1]?
Redes neurais convergem mais rápido e de forma mais estável com inputs em escala pequena. Pixels brutos (0–255) têm magnitude alta e podem causar gradientes instáveis.

### Por que data augmentation apenas no treino?
Augmentation é uma forma de regularização: cria variações artificiais para o modelo não memorizar as imagens exatas de treino. Não aplicamos no teste/validação para que a avaliação seja fiel à distribuição real.

### Por que flip horizontal e não vertical?
Mama esquerda e direita são espelhadas horizontalmente — é uma variação anatomicamente válida. Flip vertical geraria imagens impossíveis (mama de cabeça para baixo).

In [ ]:
IMG_SIZE = (224, 224)
BATCH_SIZE = 32

# Augmentation aplicada apenas no treino
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.1),  # ±10 graus
    tf.keras.layers.RandomZoom(0.1),      # ±10% zoom
], name='augmentation')

def load_and_preprocess(path, label):
    img = tf.io.read_file(path)
    # decode_image aceita PNG, JPEG, BMP e GIF — mais robusto que decode_png
    img = tf.image.decode_image(img, channels=3, expand_animations=False)
    img = tf.image.resize(img, IMG_SIZE)
    img = tf.cast(img, tf.float32) / 255.0
    img.set_shape((*IMG_SIZE, 3))  # necessário para o tf.data inferir o shape
    return img, label

def build_dataset(paths, labels, augment=False):
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    ds = ds.map(load_and_preprocess, num_parallel_calls=tf.data.AUTOTUNE)
    if augment:
        ds = ds.map(
            lambda x, y: (data_augmentation(x, training=True), y),
            num_parallel_calls=tf.data.AUTOTUNE
        )
    ds = ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
    return ds

# Extrair caminhos e rótulos
train_paths  = df_train['full_path'].values
train_labels = df_train['label'].values.astype(np.int32)
test_paths   = df_test['full_path'].values
test_labels  = df_test['label'].values.astype(np.int32)

# Verificar se os caminhos existem antes de treinar
missing = [p for p in train_paths[:5] if not os.path.exists(p)]
if missing:
    print("⚠️  Caminhos não encontrados (primeiros 5 do treino):")
    for p in missing:
        print(" ", p)
    print("\nListando estrutura real do dataset:")
    !find /content/cbis -name "*.png" -o -name "*.jpg" | head -20
else:
    print("Caminhos verificados ✓")

# Calcular class_weight para compensar desbalanceamento
class_weights_array = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(train_labels),
    y=train_labels
)
class_weight_dict = {0: class_weights_array[0], 1: class_weights_array[1]}
print(f"Class weights: Benigno={class_weight_dict[0]:.3f} | Maligno={class_weight_dict[1]:.3f}")

# Construir datasets
train_ds = build_dataset(train_paths, train_labels, augment=True)
test_ds  = build_dataset(test_paths,  test_labels,  augment=False)

print(f"\nBatches de treino: {len(train_ds)}")
print(f"Batches de teste:  {len(test_ds)}")

In [ ]:
# Visualizar efeito do data augmentation
sample_img, _ = next(iter(
    tf.data.Dataset.from_tensor_slices(
        (train_paths[:1], train_labels[:1])
    ).map(load_and_preprocess)
))

fig, axes = plt.subplots(1, 5, figsize=(15, 3))
fig.suptitle('Exemplo de Data Augmentation', fontsize=12)
axes[0].imshow(sample_img.numpy())
axes[0].set_title('Original')
axes[0].axis('off')

for i in range(1, 5):
    aug = data_augmentation(tf.expand_dims(sample_img, 0), training=True)[0]
    axes[i].imshow(aug.numpy())
    axes[i].set_title(f'Augmentada {i}')
    axes[i].axis('off')

plt.tight_layout()
plt.show()

---
## Etapa 4 — Modelo Base: CNN do Zero (Baseline)

Antes de usar transfer learning, construímos uma CNN simples para servir de **referência comparativa**. Se o EfficientNetB0 não superar este baseline, há algo errado no pipeline.

### Por que ReLU nas camadas ocultas?
- **ReLU** (*Rectified Linear Unit*) = `max(0, x)`
- Resolve o *vanishing gradient*: sua derivada é sempre 0 ou 1, então o gradiente não se dissipa ao passar por muitas camadas
- Computacionalmente eficiente: basta um limiar em zero
- Não satura para valores positivos (diferente de sigmoid e tanh)

### Por que Softmax na última camada?
- Converte os 2 logits de saída em probabilidades que **somam 1**: `[P(benigno), P(maligno)]`
- Permite interpretar a saída como confiança do modelo em cada classe
- Mais explícito e interpretável que sigmoid com 1 neurônio quando temos 2 classes

In [ ]:
def build_baseline_cnn():
    model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=(*IMG_SIZE, 3)),

        # Bloco 1: detecta bordas e texturas simples
        tf.keras.layers.Conv2D(32, (3, 3), activation='relu', padding='same'),
        tf.keras.layers.MaxPooling2D((2, 2)),

        # Bloco 2: detecta padrões mais complexos
        tf.keras.layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
        tf.keras.layers.MaxPooling2D((2, 2)),

        # Bloco 3: detecta features de alto nível
        tf.keras.layers.Conv2D(128, (3, 3), activation='relu', padding='same'),
        tf.keras.layers.MaxPooling2D((2, 2)),

        # Classificador
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(128, activation='relu'),
        tf.keras.layers.Dropout(0.5),  # regularização

        # Saída: 2 neurônios + softmax → [P(benigno), P(maligno)]
        tf.keras.layers.Dense(2, activation='softmax')
    ], name='baseline_cnn')
    return model

baseline = build_baseline_cnn()
baseline.summary()

In [ ]:
baseline.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

print("Treinando CNN baseline (10 epochs)...")
history_baseline = baseline.fit(
    train_ds,
    epochs=10,
    validation_data=test_ds,
    class_weight=class_weight_dict,
    verbose=1
)

In [ ]:
# Avaliação do baseline
y_pred_baseline = np.argmax(baseline.predict(test_ds), axis=1)

print("=" * 50)
print("BASELINE CNN — Relatório de Classificação")
print("=" * 50)
print(classification_report(
    test_labels, y_pred_baseline,
    target_names=['Benigno', 'Maligno']
))

---
## Etapa 5 — Transfer Learning com EfficientNetB0

### O que é Transfer Learning?
Transfer learning reutiliza pesos de um modelo treinado em uma tarefa grande (ImageNet: 1,2 milhões de imagens, 1000 classes) para uma tarefa menor. As camadas convolucionais aprenderam detectores genéricos de bordas, texturas e formas que funcionam em qualquer imagem, inclusive médica.

### Por que EfficientNetB0?
- **Eficiente:** escalonamento composto (largura × profundidade × resolução) para máxima performance com ~5,3M parâmetros
- **Preciso:** supera VGG e ResNet em benchmarks com menos parâmetros
- **Colab-friendly:** cabe na VRAM da GPU T4 com batch=32

### Estratégia de duas fases
1. **Fase 1 (frozen):** treinar apenas o head com a base congelada. Isso evita *catastrophic forgetting* — destruir os pesos pré-treinados com gradientes altos de um head aleatório.
2. **Fase 2 (fine-tuning):** descongelar as últimas camadas e treinar com lr muito menor, adaptando os detectores de alto nível ao domínio médico.

In [ ]:
def build_efficientnet(trainable_base=False):
    # Base pré-treinada no ImageNet (sem a camada de classificação original)
    base = tf.keras.applications.EfficientNetB0(
        include_top=False,
        weights='imagenet',
        input_shape=(*IMG_SIZE, 3)
    )
    base.trainable = trainable_base

    inputs = tf.keras.Input(shape=(*IMG_SIZE, 3))
    x = base(inputs, training=False)      # training=False mantém BatchNorm frozen
    x = tf.keras.layers.GlobalAveragePooling2D()(x)  # (7,7,1280) → (1280,)
    x = tf.keras.layers.Dropout(0.3)(x)              # regularização
    x = tf.keras.layers.Dense(128, activation='relu')(x)  # camada densa intermediária
    outputs = tf.keras.layers.Dense(2, activation='softmax')(x)  # [P(benigno), P(maligno)]

    return tf.keras.Model(inputs, outputs, name='efficientnetb0_transfer'), base

model_effnet, base_effnet = build_efficientnet(trainable_base=False)
model_effnet.summary()

print(f"\nParâmetros treináveis (Fase 1): {model_effnet.count_params():,}")
print(f"Parâmetros da base (frozen): {base_effnet.count_params():,}")

In [ ]:
# Fase 1: treinar apenas o head (base frozen)
model_effnet.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy', tf.keras.metrics.Recall(name='recall')]
)

print("Fase 1 — Treinando head (base frozen, 10 epochs)...")
history_phase1 = model_effnet.fit(
    train_ds,
    epochs=10,
    validation_data=test_ds,
    class_weight=class_weight_dict,
    verbose=1
)

In [ ]:
# Fase 2: fine-tuning — descongelar as últimas 30 camadas da base
base_effnet.trainable = True
for layer in base_effnet.layers[:-30]:
    layer.trainable = False

# Recompilar com learning rate muito menor para não destruir pesos pré-treinados
model_effnet.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy', tf.keras.metrics.Recall(name='recall')]
)

print(f"Parâmetros treináveis após descongelar: {model_effnet.count_params():,}")
print("\nFase 2 — Fine-tuning (últimas 30 camadas, 10 epochs)...")

history_phase2 = model_effnet.fit(
    train_ds,
    epochs=10,
    validation_data=test_ds,
    class_weight=class_weight_dict,
    verbose=1
)

---
## Etapa 6 — Comparação das Configurações de Loss e Métrica

Esta é a **seção central** do notebook. Treinamos a mesma arquitetura EfficientNetB0 com dois setups diferentes e comparamos o impacto na detecção de casos malignos.

### Setup A — `binary_crossentropy` + `Recall`

`binary_crossentropy` = `-[y·log(p) + (1-y)·log(1-p)]`

Penaliza previsões confiantes e erradas. Com softmax + 2 classes, usamos a probabilidade da classe positiva (maligno) diretamente.

**Recall** = `TP / (TP + FN)` — mede quantos casos malignos reais foram detectados. Em triagem médica, um **falso negativo** (maligno não detectado) pode levar a um diagnóstico tardio com consequências graves.

### Setup B — `sparse_categorical_crossentropy` + `accuracy`

`sparse_categorical_crossentropy` é a versão da categorical crossentropy onde os labels são fornecidos como inteiros (0 ou 1) em vez de one-hot encoding ([1,0] ou [0,1]). Matematicamente equivalente, mais conveniente computacionalmente.

**Accuracy** = `(TP + TN) / total` — simples, mas enganosa com classes desbalanceadas: um modelo que sempre prediz "benigno" pode ter accuracy alta sem detectar nenhum caso maligno.

In [ ]:
# ─────────────────────────────────────────────
# SETUP A: binary_crossentropy + Recall
# ─────────────────────────────────────────────
model_A, base_A = build_efficientnet(trainable_base=False)

model_A.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss='binary_crossentropy',
    metrics=[tf.keras.metrics.Recall(name='recall')]
)

# Para binary_crossentropy com softmax, precisamos converter labels para one-hot
# Alternativa: usar apenas a coluna da classe maligna (índice 1)
# Vamos criar datasets com one-hot encoding para o Setup A
def build_dataset_onehot(paths, labels, augment=False):
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    ds = ds.map(
        lambda p, l: (*load_and_preprocess(p, l)[:1],
                      tf.one_hot(load_and_preprocess(p, l)[1], depth=2)),
        num_parallel_calls=tf.data.AUTOTUNE
    )
    if augment:
        ds = ds.map(
            lambda x, y: (data_augmentation(x, training=True), y),
            num_parallel_calls=tf.data.AUTOTUNE
        )
    ds = ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
    return ds

train_ds_oh = build_dataset_onehot(train_paths, train_labels, augment=True)
test_ds_oh  = build_dataset_onehot(test_paths,  test_labels,  augment=False)

print("Treinando Setup A (binary_crossentropy + Recall, 15 epochs)...")
history_A = model_A.fit(
    train_ds_oh,
    epochs=15,
    validation_data=test_ds_oh,
    class_weight=class_weight_dict,
    verbose=1
)

In [ ]:
# ─────────────────────────────────────────────
# SETUP B: sparse_categorical_crossentropy + accuracy
# ─────────────────────────────────────────────
model_B, base_B = build_efficientnet(trainable_base=False)

model_B.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

print("Treinando Setup B (sparse_categorical_crossentropy + accuracy, 15 epochs)...")
history_B = model_B.fit(
    train_ds,
    epochs=15,
    validation_data=test_ds,
    class_weight=class_weight_dict,
    verbose=1
)

In [ ]:
# Comparação visual das curvas de treinamento
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
fig.suptitle('Comparação Setup A vs. Setup B — Curvas de Treinamento', fontsize=14)

epochs = range(1, 16)

# Gráfico 1: Loss
axes[0].plot(epochs, history_A.history['loss'],     'b-',  label='A - train loss')
axes[0].plot(epochs, history_A.history['val_loss'], 'b--', label='A - val loss')
axes[0].plot(epochs, history_B.history['loss'],     'r-',  label='B - train loss')
axes[0].plot(epochs, history_B.history['val_loss'], 'r--', label='B - val loss')
axes[0].set_title('Loss por Epoch')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Gráfico 2: Métrica principal de cada setup
axes[1].plot(epochs, history_A.history['recall'],     'b-',  label='A - train recall')
axes[1].plot(epochs, history_A.history['val_recall'], 'b--', label='A - val recall')
axes[1].plot(epochs, history_B.history['accuracy'],     'r-',  label='B - train accuracy')
axes[1].plot(epochs, history_B.history['val_accuracy'], 'r--', label='B - val accuracy')
axes[1].set_title('Métrica Principal por Epoch')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Valor')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Avaliação completa de ambos os setups no conjunto de teste
from sklearn.metrics import precision_recall_fscore_support

y_pred_A_proba = model_A.predict(test_ds_oh)[:, 1]
y_pred_A = (y_pred_A_proba >= 0.5).astype(int)

y_pred_B_proba = model_B.predict(test_ds)[:, 1]
y_pred_B = (y_pred_B_proba >= 0.5).astype(int)

def get_metrics(y_true, y_pred, y_proba):
    prec, rec, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, labels=[1], average='binary'
    )
    acc = (y_true == y_pred).mean()
    auc = roc_auc_score(y_true, y_proba)
    return {'Accuracy': acc, 'Precision (Maligno)': prec,
            'Recall (Maligno)': rec, 'F1 (Maligno)': f1, 'ROC-AUC': auc}

metrics_A = get_metrics(test_labels, y_pred_A, y_pred_A_proba)
metrics_B = get_metrics(test_labels, y_pred_B, y_pred_B_proba)

comparison_df = pd.DataFrame({
    'Setup A (binary_CE + Recall)': metrics_A,
    'Setup B (sparse_cat_CE + Acc)': metrics_B
}).round(4)

print("=" * 60)
print("TABELA COMPARATIVA — Setup A vs. Setup B")
print("=" * 60)
print(comparison_df.to_string())
print("\n→ O Recall da classe maligna é a métrica mais importante aqui.")
print("  Um Recall baixo significa casos malignos não detectados (alto risco clínico).")

---
## Etapa 7 — Avaliação Final no Conjunto de Teste

Avaliação detalhada do **melhor modelo (Setup A)** com todas as métricas clínicas relevantes.

In [ ]:
print("SETUP A — Relatório de Classificação Completo")
print("=" * 55)
print(classification_report(
    test_labels, y_pred_A,
    target_names=['Benigno', 'Maligno']
))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Matriz de confusão
cm = confusion_matrix(test_labels, y_pred_A)
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Blues',
    xticklabels=['Benigno', 'Maligno'],
    yticklabels=['Benigno', 'Maligno'],
    ax=axes[0]
)
axes[0].set_title('Matriz de Confusão — Setup A', fontsize=12)
axes[0].set_xlabel('Predito')
axes[0].set_ylabel('Real')

tn, fp, fn, tp = cm.ravel()
axes[0].text(0.5, -0.15,
    f'VP={tp} | FP={fp} | FN={fn} | VN={tn}\n'
    f'Falsos Negativos (malignos não detectados): {fn}',
    transform=axes[0].transAxes, ha='center', fontsize=9, color='darkred')

# Curva ROC
fpr, tpr, _ = roc_curve(test_labels, y_pred_A_proba)
auc_score = roc_auc_score(test_labels, y_pred_A_proba)
axes[1].plot(fpr, tpr, color='steelblue', lw=2, label=f'ROC-AUC = {auc_score:.3f}')
axes[1].plot([0,1], [0,1], 'k--', label='Classificador aleatório')
axes[1].fill_between(fpr, tpr, alpha=0.1, color='steelblue')
axes[1].set_xlabel('Taxa de Falsos Positivos (1 - Especificidade)')
axes[1].set_ylabel('Taxa de Verdadeiros Positivos (Sensibilidade)')
axes[1].set_title('Curva ROC — Setup A', fontsize=12)
axes[1].legend(loc='lower right')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---
## Etapa 8 — Grad-CAM: Visualização de Ativações

**Grad-CAM** (*Gradient-weighted Class Activation Mapping*) é uma técnica de interpretabilidade que mostra **quais regiões da imagem mais influenciaram a predição do modelo**.

**Como funciona:**
1. Faz uma passagem forward pela rede e registra os mapas de ativação da última camada convolucional
2. Calcula o gradiente da classe predita em relação a esses mapas de ativação
3. Usa a média dos gradientes como pesos para ponderar cada mapa de ativação
4. Aplica ReLU no resultado (só ativações positivas importam) e redimensiona para o tamanho original

**Por que é importante academicamente:** prova que o modelo aprendeu padrões nas regiões de lesão e não fez "trapaça" com artefatos de fundo. É um requisito para confiança clínica.

In [ ]:
import cv2

def make_gradcam_heatmap(img_array, model, last_conv_layer_name, pred_index=None):
    # Criar modelo que retorna: ativações da última conv + predições
    grad_model = tf.keras.Model(
        inputs=model.inputs,
        outputs=[model.get_layer(last_conv_layer_name).output, model.output]
    )

    with tf.GradientTape() as tape:
        conv_outputs, predictions = grad_model(img_array)
        if pred_index is None:
            pred_index = tf.argmax(predictions[0])
        class_channel = predictions[:, pred_index]

    # Gradiente da classe predita em relação aos mapas de ativação
    grads = tape.gradient(class_channel, conv_outputs)

    # Média dos gradientes por canal (pesos de importância)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))

    # Ponderar mapas de ativação pelos pesos
    conv_outputs = conv_outputs[0]
    heatmap = conv_outputs @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)
    heatmap = tf.maximum(heatmap, 0) / (tf.math.reduce_max(heatmap) + 1e-8)
    return heatmap.numpy()

def overlay_gradcam(img_path, heatmap, alpha=0.4):
    img = cv2.imread(img_path)
    img = cv2.resize(img, IMG_SIZE)
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    heatmap_resized = cv2.resize(heatmap, IMG_SIZE)
    heatmap_colored = cv2.applyColorMap(
        np.uint8(255 * heatmap_resized), cv2.COLORMAP_JET
    )
    heatmap_rgb = cv2.cvtColor(heatmap_colored, cv2.COLOR_BGR2RGB)
    superimposed = cv2.addWeighted(img_rgb, 1 - alpha, heatmap_rgb, alpha, 0)
    return img_rgb, superimposed

# Identificar a última camada convolucional do EfficientNetB0
# (dentro do modelo A)
effnet_base = model_A.layers[1]  # camada do EfficientNetB0
last_conv = [l.name for l in effnet_base.layers if 'conv' in l.name.lower()][-1]
print(f"Última camada convolucional: {last_conv}")

In [ ]:
# Selecionar amostras para Grad-CAM: 2 corretas + 2 incorretas
indices_malignos = np.where(test_labels == 1)[0]
acertos = [i for i in indices_malignos if y_pred_A[i] == 1][:2]
erros   = [i for i in indices_malignos if y_pred_A[i] == 0][:2]
samples = (
    [(i, 'Correto (Maligno → Maligno)') for i in acertos] +
    [(i, 'Erro (Maligno → Benigno)')    for i in erros]
)

fig, axes = plt.subplots(len(samples), 2, figsize=(10, 4 * len(samples)))
fig.suptitle('Grad-CAM — Regiões de Ativação do Modelo', fontsize=14)

for row, (idx, title) in enumerate(samples):
    img_path = test_paths[idx]
    img_arr = load_and_preprocess(img_path, 0)[0]
    img_input = tf.expand_dims(img_arr, 0)

    heatmap = make_gradcam_heatmap(img_input, model_A, last_conv, pred_index=1)
    original, overlaid = overlay_gradcam(img_path, heatmap)

    color = 'green' if 'Correto' in title else 'red'
    axes[row][0].imshow(original)
    axes[row][0].set_title(f'Original\n{title}', color=color)
    axes[row][0].axis('off')

    axes[row][1].imshow(overlaid)
    axes[row][1].set_title('Grad-CAM (regiões quentes = mais relevantes)')
    axes[row][1].axis('off')

plt.tight_layout()
plt.show()

---
## Etapa 9 — Discussão Crítica

### 9.1 — Por que ReLU e Softmax?

**ReLU nas camadas ocultas:**  
A função `max(0, x)` resolve o *vanishing gradient* que afetava redes profundas com sigmoid/tanh: como sua derivada é 0 ou 1 (nunca satura positivamente), o gradiente consegue se propagar por muitas camadas sem se extinguir. Além disso, é computacionalmente barata e introduz a não-linearidade necessária para que a rede aprenda representações complexas.

**Softmax na camada final:**  
Transforma os logits de saída em uma distribuição de probabilidade válida (`sum = 1`). Com 2 neurônios, a saída é `[P(benigno), P(maligno)]`, o que é intuitivo e interpretável. A alternativa (sigmoid + 1 neurônio) daria o mesmo resultado numérico, mas é menos explícita quando há mais de 2 classes — manter softmax é uma escolha de generalidade.

---

### 9.2 — Diferença Prática entre as Duas Loss Functions

**`binary_crossentropy`** trata cada classe independentemente e penaliza a diferença entre a probabilidade predita e o label real. Com class_weight, o termo da classe maligna recebe peso maior, pressionando o modelo a não errar malignos.

**`sparse_categorical_crossentropy`** minimiza a entropia cruzada entre a distribuição predita e o label correto. É matematicamente equivalente à categorical crossentropy, mas aceita labels inteiros em vez de one-hot, economizando memória.

A diferença principal **não é matemática**, mas de **sinal de treinamento**: monitorar `Recall` durante o treino guia o otimizador a favorecer verdadeiros positivos, enquanto `accuracy` pode ser maximizada ignorando a classe minoritária.

---

### 9.3 — Limitações do Dataset CBIS-DDSM

1. **Viés temporal:** imagens coletadas nos anos 1990 com equipamentos de mamografia da época. Mamógrafos modernos têm resolução e contraste diferentes.
2. **Viés de população:** pacientes majoritariamente de hospitais americanos, com distribuição de idade e histórico familiar não representativa de outras populações.
3. **Variabilidade de anotação:** os labels foram definidos por radiologistas, e há variação inter-observador — especialmente na categoria `BENIGN_WITHOUT_CALLBACK`.
4. **Tamanho limitado:** ~3.500 casos para treino é pequeno para deep learning. O modelo depende fortemente do transfer learning pré-treinado.

---

### 9.4 — Desbalanceamento de Classes

O dataset tem mais casos benignos que malignos. Sem correção, o modelo aprende a "chutar" benigno e obtém accuracy artificialmente alta. A estratégia usada foi `class_weight='balanced'`, que aumenta o custo de errar a classe minoritária (maligno) durante o treinamento.

Alternativas não exploradas aqui: *oversampling* com SMOTE (inviável para imagens), coleta de mais dados malignos, ou *threshold adjustment* (reduzir o limiar de 0.5 para aumentar Recall).

---

### 9.5 — Aplicação Clínica Real

Para uso clínico, um modelo deste tipo precisaria de:
- **Validação prospectiva** em pacientes reais com acompanhamento histopatológico
- **Aprovação regulatória** (ANVISA no Brasil, FDA nos EUA) como dispositivo médico de software (SaMD)
- **Rastreabilidade de decisões** — Grad-CAM é um passo, mas não suficiente para auditoria
- **Integração com workflow clínico** sem substituir o radiologista, mas sim priorizando casos para revisão

---

### 9.6 — Próximos Passos

- **Dataset maior:** VinDr-Mammo (~20.000 casos), INbreast, ou Duke Breast Cancer MRI
- **Arquitetura maior:** EfficientNetB4 ou ViT (Vision Transformer) para maior capacidade
- **Ensemble:** combinar predições de modelos treinados em diferentes splits
- **Segmentação:** detectar e localizar a lesão antes de classificar (pipeline de dois estágios)
- **Validação externa:** treinar em CBIS-DDSM e testar em INbreast para avaliar generalização

In [ ]:
# Resumo final dos resultados
print("=" * 65)
print("RESUMO FINAL")
print("=" * 65)
print(f"\nModelo: EfficientNetB0 + Transfer Learning")
print(f"Dataset: CBIS-DDSM ({len(df_train)} treino | {len(df_test)} teste)")
print()
print(comparison_df.to_string())
print()
print("Conclusão:")
print("  O Setup A (binary_crossentropy + Recall) é mais adequado para triagem")
print("  médica porque prioriza a detecção de casos malignos (minimiza FN).")
print("  O Setup B (sparse_categorical_crossentropy + accuracy) é mais intuitivo")
print("  de interpretar, mas pode subestimar casos malignos em datasets desbalanceados.")